<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.1-stationary-heat/Ex08.1_03_weights_and_flux_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.1 · Notebook 03 — Weighting the Flux Term, and the Flux Balance

**Paired with L8.1 · Stationary Heat Transfer**

Two checks that a contour plot cannot give you.

The first is diagnostic: plot the loss terms **separately**. If the total is
falling while the flux term sits flat, the flux condition is being ignored —
slide 11, and the reason Liu departs from his `w = 1` default on exactly this
problem and sets $w^{NBC} = 100$.

The second is physical: in steady state everything generated must leave. That
is an equation you can check without knowing the answer, which makes it worth
more than an error norm.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.1-stationary-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · Sweep the flux weight

### TODO 1 — sweep the flux weight

Re-run the plate problem for `W_FLUX` in 1, 10, 100, recording the PDE and
flux losses **separately** each time. Copy your `trial`, `residual` and
`flux_loss` from Notebook 02 — retyping them is how you find out whether you
understood them.

The points are sampled once, outside the loop, so every weight sees the same
sample and the comparison is about the weight and nothing else.

In [ ]:
Q_OVER_K = 10.0
xy_f = to_tensor(pb.sample_plate_with_hole(1500), requires_grad=True)
xy_o = to_tensor(pb.sample_outer_edges(30), requires_grad=True)

# TODO 1 --- three flux weights -----------------------------------------------------------------------------
# Two `...` to replace, inside the loop:
#   line 1  ->  mse(residual(m, xy_f)) + w * flux_loss(m, xy_o)      the weighted loss for this w
#   line 2  ->  float(flux_loss(m, xy_o))                             the bare flux term, recorded UNWEIGHTED
# trial / residual / flux_loss are the ones from notebook 02, without the weight.
def trial(model, xy):
    return pb.hole_multiplier(xy) * model(xy)

def residual(model, xy):
    T = trial(model, xy)
    return d2(T, xy, 0) + d2(T, xy, 1) + Q_OVER_K

def flux_loss(model, xy):
    g = grad(trial(model, xy), xy)
    return (g ** 2).sum(dim=1, keepdim=True).mean()

results = {}
for w in (1.0, 10.0, 100.0):
    set_seed(88)
    m = MLP(n_in=2, n_hidden=40, n_layers=3)

    def loss_fn(m=m, w=w):
        return ...                                # <- mse(residual(m, xy_f)) + w * flux_loss(m, xy_o)

    train_two_stage(m, loss_fn, adam_steps=1500, lbfgs_steps=80, lr=1e-3, report_every=0)
    pde_loss_final  = float(mse(residual(m, xy_f)))
    flux_loss_final = ...                         # <- float(flux_loss(m, xy_o))
    results[w] = (pde_loss_final, flux_loss_final)
    print(f"w = {w:g}: PDE {pde_loss_final:.3e}   flux {flux_loss_final:.3e}")
# ------------------------------------------------------------------------------

In [ ]:
WEIGHTS = sorted(results)
pde = [results[w][0] for w in WEIGHTS]
flx = [results[w][1] for w in WEIGHTS]

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(WEIGHTS, pde, "o-", lw=1.9, ms=6, color=CYCLE[1], label="PDE residual")
ax.plot(WEIGHTS, flx, "s--", lw=1.9, ms=6, color=CYCLE[2], label="flux term")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("flux weight  w"); ax.set_ylabel("unweighted loss term")
ax.set_title("What the flux weight buys, and what it costs")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

print(error_table(
    [[f"{w:g}", f"{p:.3e}", f"{f:.3e}"] for w, p, f in zip(WEIGHTS, pde, flx)],
    ["w", "PDE residual", "flux term"]))

## 2 · The flux balance

### TODO 2 — the flux balance

In steady state, heat leaving through the hole must equal heat generated in
the plate. `flux_balance` returns both; compare them.

`xy_h` must carry `requires_grad=True`: the normal flux is a derivative of the
network, so `flux_balance` differentiates at those points. Without it `grad`
returns `None` and the call fails.

In [ ]:
xy_h = to_tensor(pb.sample_ellipse_boundary(200), requires_grad=True)

# TODO 2 --- the flux balance ------------------------------------------------------------------------------------
# One `...` to replace:  pb.flux_balance(m, xy_h, Q_OVER_K, trial=trial)
#   heat leaving through the hole must equal the heat generated in the plate
out, gen = ...                                    # <- pb.flux_balance(m, xy_h, Q_OVER_K, trial=trial)
print(f"out {out:.4f}  generated {gen:.4f}  mismatch {abs(out-gen)/gen:.2%}")
# ------------------------------------------------------------------------------

A mismatch above a few percent means the flux condition was never
learned.

Note what the two sides are made of. `generated` is pure geometry —
`Q_OVER_K` times the material area, `0.93780`, and it does not involve the
network at all. `out` is the mean normal flux on the hole times the hole
perimeter, `0.92438`, and it is entirely the network's derivative. They have no
reason to agree unless the physics was learned.

---

## 3 · Save

In [ ]:
os.makedirs("Ex08.1_outputs", exist_ok=True)
path = os.path.join("Ex08.1_outputs", "nb03_flux.npz")
np.savez(path,
         weights=np.asarray(WEIGHTS, dtype=float),
         pde=np.asarray(pde, dtype=float),
         flux=np.asarray(flx, dtype=float),
         out=float(out), generated=float(gen),
         mismatch=float(abs(out - gen) / gen))
print("wrote", path)

## 4 · Before you move on

1. Which weight gave the smallest flux term, and which gave the smallest PDE
   residual? If they are not the same weight, say what you would choose and on
   what grounds.
2. The flux balance uses the mean normal flux times the perimeter. Name the
   assumption that hides in the word *mean*, and say when it would fail.
3. Your mismatch is some percentage. State whether you would sign off a design
   on it, and what you would measure next if you would not.

Next: **notebook 04**, the comparison and the report.